[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto3/3pipeline.ipynb)
Confirmed runtime version: 2026.04

## Setup

In [ ]:
import sys

vi = sys.version_info
if not ((3, 12) <= (vi.major, vi.minor) < (3, 13)):
    raise RuntimeError(f"Python 3.12 required, got {vi.major}.{vi.minor}.{vi.micro}")

print(f"Python {vi.major}.{vi.minor}.{vi.micro} ✓")

In [ ]:
!pip install -q google-genai

In [ ]:
import json

print("Setup complete.")

## Data Models

In [ ]:
ROLES = ["TechnicalMethod", "Task", "Dataset", "EvaluationMetric"]

print("Models ready.")

## Stage 0 — Parse XML

In [ ]:
from xml.etree import ElementTree as ET

from google.colab import files

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
SKIP_HEADINGS = {"references", "acknowledgements", "acknowledgments"}


def _text(element) -> str:
    return " ".join(element.itertext()).strip()


uploaded = files.upload()
xml_filename = next(iter(uploaded))
xml_bytes = uploaded[xml_filename]

root = ET.fromstring(xml_bytes.decode("utf-8"))

abstract_el = root.find(".//tei:abstract", NS)
abstract_text = _text(abstract_el) if abstract_el is not None else ""

sections = []

for div in root.findall(".//tei:body//tei:div", NS):
    heading = div.findtext("tei:head", namespaces=NS) or ""
    h_lower = heading.lower().strip()

    if h_lower in SKIP_HEADINGS:
        continue

    body = " ".join(_text(p) for p in div.findall("tei:p", NS)).strip()
    if body:
        sections.append({"heading": heading, "text": body})

if abstract_text:
    sections.insert(0, {"heading": "Abstract", "text": abstract_text})

print(f"Loaded: {xml_filename}")
print(f"Sections: {len(sections)}")
for s in sections:
    print(f"  - {s['heading']}")

## Stage 1 — Text Extraction

In [ ]:
document_text = ""
for s in sections:
    document_text += f"## {s['heading']}\n\n{s['text']}\n\n"

print(f"Document length: {len(document_text)} characters")
print(document_text[:500])

## Stage 2 — LLM Extraction (Gemini)

Before running this section, add your API key as a Colab secret: click the key icon in the left sidebar, add a secret named `GEMINI_API_KEY`, and enable notebook access.

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL_NAME = "gemini-2.5-flash"

print("Gemini client ready.")

## Stage 2b — Prompt Template

In [ ]:
PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "For each of the four roles below, return:\n"
    '- "answer": the shortest identifying term (e.g. "Transformer", not '
    '"a novel attention-based model")\n'
    '- "evidence": one sentence quoted directly from the paper that '
    "supports the answer\n"
    "\n"
    "Roles:\n"
    "- TechnicalMethod: the main method, model, algorithm, or system "
    "proposed by the authors\n"
    "- Task: the research task or problem being addressed\n"
    "- Dataset: the dataset used for training or evaluation\n"
    "- EvaluationMetric: the metric used to report results\n"
    "\n"
    "Rules:\n"
    "- Use the authors' own method, not methods cited from prior work.\n"
    "- If a field is not present in the paper, return null for both "
    "answer and evidence.\n"
    "- For each evidence, return the section heading and one sentence "
    "quoted verbatim from the paper.\n"
    "- Return only the JSON object, no explanation.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

prompt = PROMPT_TEMPLATE.format(paper_text=document_text)
print(f"Prompt length: {len(prompt)} characters")

## Stage 2c — Call Gemini and Parse Response

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
)

raw_text = response.text.strip()
if raw_text.startswith("```"):
    raw_text = raw_text.strip("`")
    raw_text = raw_text.removeprefix("json").strip()

profile = json.loads(raw_text)

for role in ROLES:
    entry = profile.get(role)
    print(f"{role}: {entry}")

print()
print(json.dumps(profile, indent=2, ensure_ascii=False))